In [1]:
import pandas as pd
import json
import sqlite3
import os

# Set pandas to display all columns for better inspection
pd.set_option('display.max_columns', None)

In [2]:
# Load Transactional Data (CSV)
orders = pd.read_csv('orders.csv')

# Load User Master Data (JSON)
with open('users.json', 'r') as f:
    users_list = json.load(f)
users = pd.DataFrame(users_list)

print(f"Orders Loaded: {len(orders)} rows")
print(f"Users Loaded: {len(users)} rows")

Orders Loaded: 10000 rows
Users Loaded: 3000 rows


In [3]:
# Create a temporary in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Read and execute the SQL script
with open('restaurants.sql', 'r') as f:
    sql_script = f.read()
conn.executescript(sql_script)

# Load into a Pandas DataFrame
restaurants = pd.read_sql('SELECT * FROM restaurants', conn)
print(f"Restaurants Loaded: {len(restaurants)} rows")

Restaurants Loaded: 500 rows


In [4]:
# Perform Left Joins
master_df = pd.merge(orders, users, on='user_id', how='left')
master_df = pd.merge(master_df, restaurants, on='restaurant_id', how='left')

# Data Cleaning
# 1. Convert order_date to datetime (handling Day-First format)
master_df['order_date'] = pd.to_datetime(master_df['order_date'], dayfirst=True)

# 2. Ensure total_amount is numeric
master_df['total_amount'] = pd.to_numeric(master_df['total_amount'], errors='coerce')

# Save the final dataset
master_df.to_csv('final_food_delivery_dataset.csv', index=False)
print("Final dataset created: 'final_food_delivery_dataset.csv'")

Final dataset created: 'final_food_delivery_dataset.csv'


In [6]:
print("--- MCQ Analysis ---")

# Q1: Highest Revenue City for Gold Members
q1 = master_df[master_df['membership'] == 'Gold'].groupby('city')['total_amount'].sum().idxmax()
print(f"Q1 (Top Gold City): {q1}")

# Q2: Highest Avg Order Value by Cuisine
q2 = master_df.groupby('cuisine')['total_amount'].mean().idxmax()
print(f"Q2 (Top Avg Cuisine): {q2}")

# Q4: Revenue by Rating Range
bins = [3.0, 3.5, 4.0, 4.5, 5.1]
labels = ['3.0 – 3.5', '3.6 – 4.0', '4.1 – 4.5', '4.6 – 5.0']
master_df['rating_range'] = pd.cut(master_df['rating'], bins=bins, labels=labels, right=False)
q4 = master_df.groupby('rating_range')['total_amount'].sum().idxmax()
print(f"Q4 (Top Rating Range): {q4}")

# Q7: Percentage of Gold Member Orders
gold_pct = (len(master_df[master_df['membership'] == 'Gold']) / len(master_df)) * 100
print(f"Q7 (Gold Order %): {round(gold_pct)}%")

# Q10: Highest Revenue Quarter
master_df['quarter'] = master_df['order_date'].dt.to_period('Q')
q10 = master_df.groupby('quarter')['total_amount'].sum().idxmax()
print(f"Q10 (Top Quarter): {q10}")

--- MCQ Analysis ---
Q1 (Top Gold City): Chennai
Q2 (Top Avg Cuisine): Mexican
Q4 (Top Rating Range): 4.6 – 5.0
Q7 (Gold Order %): 50%
Q10 (Top Quarter): 2023Q3


C:\Users\Mopur Shankar Reddy\AppData\Local\Temp\ipykernel_24068\188770511.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q4 = master_df.groupby('rating_range')['total_amount'].sum().idxmax()


In [7]:
print("--- Numerical Answers ---")

# 1. Total Gold Orders
print(f"Total Gold Orders: {len(master_df[master_df['membership'] == 'Gold'])}")

# 2. Total Revenue Hyderabad
hyd_rev = master_df[master_df['city'] == 'Hyderabad']['total_amount'].sum()
print(f"Hyderabad Total Revenue: {round(hyd_rev)}")

# 3. Distinct Users
print(f"Distinct Users: {master_df['user_id'].nunique()}")

# 4. Avg Order Value Gold
gold_avg = master_df[master_df['membership'] == 'Gold']['total_amount'].mean()
print(f"Gold Avg Order Value: {round(gold_avg, 2)}")

# 5. Rating >= 4.5 Orders
print(f"Orders (Rating >= 4.5): {len(master_df[master_df['rating'] >= 4.5])}")

--- Numerical Answers ---
Total Gold Orders: 4987
Hyderabad Total Revenue: 1889367
Distinct Users: 2883
Gold Avg Order Value: 797.15
Orders (Rating >= 4.5): 3374
